# 02 · Journeys and Friction (All data)

This notebook reconstructs sessions/journeys, identifies frequent subpaths, and surfaces friction hotspots and likely root causes.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from analysis.utils import load_all, add_time_features, sessionize

dfs = load_all()
events = dfs['events']
subpaths_export = dfs['subpaths']
friction_export = dfs['friction']

# TIMEZONE CORRECTION: Add 12 hours to correct database timestamp offset
events['ts'] = events['ts'] + pd.Timedelta(hours=12)

# Time features + sessionization
events = add_time_features(events)
events = sessionize(events, gap_minutes=30)

# Basic session KPIs
session_kpis = {
    'sessions_total': events['session_id'].nunique(),
    'median_session_len': events.groupby('session_id').size().median(),
    'median_session_duration_min': (events.groupby('session_id')['ts'].max() - events.groupby('session_id')['ts'].min()).dt.total_seconds().median()/60,
}
session_kpis


{'sessions_total': 721,
 'median_session_len': np.float64(5.0),
 'median_session_duration_min': np.float64(0.22858333333333333)}

In [ ]:
# Frequent subpaths from reconstructed sessions (length 2-4)
from collections import Counter

paths = events[['session_id','url_path','ts']].dropna(subset=['url_path']).sort_values(['session_id','ts'])
sub_counts = Counter()
for sid, grp in paths.groupby('session_id'):
    seq = grp['url_path'].tolist()
    n = len(seq)
    for L in (2,3,4):
        for i in range(0, max(0, n-L+1)):
            sub_counts[tuple(seq[i:i+L])] += 1

reconstructed_subpaths = (
    pd.DataFrame([
        {'sequence': ' → '.join(k), 'length': len(k), 'count': v}
        for k, v in sub_counts.items()
    ])
    .sort_values(['length','count'], ascending=[True, False])
)
reconstructed_subpaths.head(20)


,sequence,length,count
17,http://localhost:3000/dashboard → http://local...,2,1272
312,https://www.linkedin.com/feed/ → https://www.l...,2,466
187,https://github.com/OmMistry25 → https://github...,2,335
267,https://www.youtube.com/ → https://www.youtube...,2,260
136,https://www.google.com/search → https://www.go...,2,248
924,https://canvas.illinois.edu/courses/58390/assi...,2,169
1984,https://canvas.illinois.edu/courses/58390/modu...,2,149
2994,https://www.enterprise.com/en/reserve.html → h...,2,144
1367,https://sonatic.com/ → https://sonatic.com/,2,141
409,http://localhost:3000/patterns → http://localh...,2,125


In [ ]:
# Friction hotspots by path/domain
friction_cols = []
if 'type' in events.columns:
    friction_cols.append('type')

# Define friction flag from event type or meta if present
def is_friction(row):
    t = str(row.get('type','')).lower()
    if 'friction' in t or 'rage' in t:
        return True
    return False

E = events.copy()
E['is_friction'] = E.apply(is_friction, axis=1)

by_path = (E.groupby('url_path')
           .agg(events=('id','count'), friction_events=('is_friction','sum'),
                last_ts=('ts','max'))
           .assign(friction_rate=lambda d: d['friction_events']/d['events'])
           .sort_values('friction_rate', ascending=False))
by_path.head(20)


,events,friction_events,last_ts,friction_rate
url_path,,,,
https://www.netflix.com/search,24,24,2025-10-13 10:08:00.312000+00:00,1.000000
https://github.com/OmMistry25/observe_and_create/commit/a3b11f022dac4c70c0a3588cdfd1ca3eef6d4380,13,13,2025-10-14 14:50:51.798000+00:00,1.000000
https://www.linkedin.com/in/bhatakash/,29,29,2025-10-13 11:02:23.038000+00:00,1.000000
https://www.linkedin.com/in/carolvc/,3,3,2025-10-20 15:23:37.146000+00:00,1.000000
https://www.linkedin.com/in/adamtheyounes/,15,15,2025-10-15 04:05:41.216000+00:00,1.000000
https://outlook.live.com/mail/0/#,1,1,2025-10-12 16:23:38.465000+00:00,1.000000
https://www.linkedin.com/search/results/all/,69,68,2025-10-14 15:09:57.702000+00:00,0.985507
https://canvas.illinois.edu/courses/59640/grades,17,16,2025-10-13 14:04:07.752000+00:00,0.941176
https://vercel.com/ommistry25s-projects/observe_and_create/settings/security,16,15,2025-10-29 05:38:31.403000+00:00,0.937500
